# Lab 10 — Métodos de Propagación de Etiquetas
**Grupo 03** | Minería de Datos — Sección 20  
Universidad del Valle de Guatemala

**Integrantes:** Felipe Aguilar, Vianka Castro, Nicolás Concuá, Ricardo Godínez, Fernando Hernández, Fernando Rueda

---

## Fase 2 — Selección y Análisis de Dataset

**Dataset:** Red Wine Quality  
**Fuente:** UCI Machine Learning Repository / Kaggle  
**Algoritmos a aplicar:** Label Propagation & Label Spreading

---
## 2a. Importación del Dataset

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('winequality-red.csv', sep=';')

print('Dataset cargado exitosamente.')
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')

In [ ]:
df.head()

In [ ]:
print('Columnas del dataset:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
feature_cols = [c for c in df.columns if c != 'quality']
X = df[feature_cols]
y_raw = df['quality']

df['quality_label'] = (df['quality'] >= 7).astype(int)
df['quality_str']   = df['quality_label'].map({0: 'bad', 1: 'good'})
y = df['quality_label']

print(f'Features : {len(feature_cols)}')
print(f'Target   : quality binarizado (>=7 = good, <7 = bad)')
print(f'Muestras : {len(df)}')

---
## 2b. Análisis Exploratorio de Datos (EDA)

### Tipos de variables y dimensionalidad

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

In [ ]:
type_summary = pd.DataFrame({
    'dtype'      : df[feature_cols].dtypes,
    'non_null'   : df[feature_cols].notna().sum(),
    'null'       : df[feature_cols].isna().sum(),
    'unique_vals': df[feature_cols].nunique()
})

print('=== Resumen de tipos de variables ===')
print(type_summary.to_string())
print(f'\nFeatures numericas  : {df[feature_cols].select_dtypes(include=np.number).shape[1]}')
print(f'Features categoricas: {df[feature_cols].select_dtypes(include="object").shape[1]}')

In [ ]:
print(f'Dimensionalidad del espacio de features: {X.shape[1]} dimensiones')
print(f'Numero de muestras                     : {X.shape[0]}')
print(f'\nDistribucion de scores de calidad originales:')
print(y_raw.value_counts().sort_index().to_string())

### Balance de clases

In [ ]:
class_counts = df['quality_str'].value_counts()
class_pct    = df['quality_str'].value_counts(normalize=True) * 100

print('=== Balance de clases (binarizado) ===')
for cls in class_counts.index:
    print(f'  {cls:6s}: {class_counts[cls]:5d} muestras ({class_pct[cls]:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
palette = {'bad': '#e74c3c', 'good': '#2ecc71'}

score_counts = y_raw.value_counts().sort_index()
axes[0].bar(score_counts.index, score_counts.values, color='#8e44ad', edgecolor='white', linewidth=0.8)
axes[0].axvline(x=6.5, color='red', linestyle='--', linewidth=1.5, label='umbral binarizacion (7)')
axes[0].set_title('Distribucion scores originales')
axes[0].set_xlabel('Quality score')
axes[0].set_ylabel('Cantidad')
axes[0].legend(fontsize=8)
for k, v in score_counts.items():
    axes[0].text(k, v + 5, str(v), ha='center', fontsize=9)

sns.countplot(data=df, x='quality_str', ax=axes[1], order=['bad', 'good'], palette=palette)
axes[1].set_title('Distribucion clases (binarizado)')
axes[1].set_xlabel('Calidad')
axes[1].set_ylabel('Cantidad')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

axes[2].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90, explode=(0, 0.05))
axes[2].set_title('Proporcion de clases')

plt.suptitle('Balance de clases — Red Wine Quality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Estadísticas descriptivas

In [ ]:
desc = X.describe().T
desc['cv'] = (desc['std'] / desc['mean']).round(3)
desc = desc[['mean', 'std', 'cv', 'min', '25%', '50%', '75%', 'max']]
desc.columns = ['Media', 'Std', 'CV', 'Min', 'Q1', 'Mediana', 'Q3', 'Max']

print('=== Estadisticas descriptivas (features) ===')
desc.round(4)

### Distribución de features por clase

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    sns.histplot(data=df, x=feat, hue='quality_str',
                 ax=axes[i], kde=True, palette=palette, alpha=0.55, bins=30)
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel('')
    if i != 0:
        axes[i].get_legend().remove()

axes[-1].set_visible(False)
plt.suptitle('Distribucion de features por clase', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Boxplots por clase

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    sns.boxplot(data=df, x='quality_str', y=feat,
                ax=axes[i], palette=palette, order=['bad', 'good'])
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
plt.suptitle('Boxplots por clase', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Matriz de correlación

In [ ]:
corr_matrix = X.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.5)
ax.set_title('Matriz de correlacion — features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r, 3)))

print('Pares con |r| > 0.70:')
for a, b, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
    print(f'  {a:30s} <-> {b:30s}  r = {r}')

### Correlación de cada feature con el score de calidad

In [ ]:
corr_with_target = X.corrwith(y_raw).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlacion de Pearson con quality')
ax.set_title('Correlacion de features con el score de calidad', fontsize=12, fontweight='bold')
for feat, val in corr_with_target.items():
    ax.text(val + (0.005 if val >= 0 else -0.005), list(corr_with_target.index).index(feat),
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

### Detección de outliers (IQR)

In [ ]:
Q1  = X.quantile(0.25)
Q3  = X.quantile(0.75)
IQR = Q3 - Q1

outlier_mask   = (X < (Q1 - 1.5 * IQR)) | (X > (Q3 + 1.5 * IQR))
outlier_counts = outlier_mask.sum().sort_values(ascending=False)

print('=== Outliers por feature (criterio IQR) ===')
print(outlier_counts[outlier_counts > 0].to_string())
print(f'\nMuestras con al menos 1 outlier: {outlier_mask.any(axis=1).sum()} / {len(df)}')

top5 = outlier_counts.head(5).index.tolist()
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, feat in enumerate(top5):
    sns.boxplot(data=df, y=feat, x='quality_str',
                ax=axes[i], palette=palette, order=['bad', 'good'])
    axes[i].set_title(feat[:20], fontsize=9)
    axes[i].set_xlabel('')

plt.suptitle('Top 5 features con mas outliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Proyección PCA — separabilidad visual

In [ ]:
from sklearn.decomposition import PCA
from matplotlib.patches import Patch

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

colors = y.map({0: '#e74c3c', 1: '#2ecc71'})
legend_elements = [Patch(facecolor='#e74c3c', label='bad  (<7)'),
                   Patch(facecolor='#2ecc71', label='good (>=7)')]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.5,
           edgecolors='k', linewidths=0.2, s=30)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Proyeccion PCA — Red Wine Quality', fontsize=12, fontweight='bold')
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

print(f'Varianza explicada acumulada: {sum(pca.explained_variance_ratio_)*100:.2f}%')

### Valores faltantes

In [ ]:
missing = X.isna().sum()
print('=== Valores faltantes por feature ===')
if missing.sum() == 0:
    print('  No se encontraron valores faltantes. OK')
else:
    print(missing[missing > 0])

---
## 2c–2d. Preprocesamiento de Datos

Los kernels RBF y k-NN de Label Propagation/Spreading dependen de **distancias euclidianas**. Las 11 features del dataset tienen rangos muy distintos (p.ej. `total sulfur dioxide` llega a ~289 mientras `density` ronda 0.99), lo que distorsionaría la matriz de afinidad W sin estandarización.

Pasos aplicados:
1. **Variables categóricas**: ninguna — no se requiere encoding.
2. **Outliers**: se conservan; los métodos de propagación son robustos a ruido moderado y eliminarlos reduciría muestras valiosas de la clase minoritaria (`good`).
3. **Estandarización z-score** (`StandardScaler`): media 0, std 1 en cada feature.
4. **Construcción del conjunto semi-supervisado**: 20% etiquetado / 80% con `y = -1`.

### Verificación de variables categóricas

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

cat_cols = X.select_dtypes(include='object').columns.tolist()
if len(cat_cols) == 0:
    print('No existen variables categoricas. One-hot encoding no requerido. OK')
else:
    print(f'Variables categoricas encontradas: {cat_cols}')

### Decisión sobre outliers

In [ ]:
outliers_por_clase = pd.DataFrame({
    'total_outliers': outlier_mask.sum(axis=1),
    'clase': df['quality_str']
})

print('Outliers por clase:')
print(outliers_por_clase.groupby('clase')['total_outliers'].describe().round(2))
print()
print('Muestras de clase "good" con outliers:',
      outlier_mask.any(axis=1)[y == 1].sum(), '/', (y == 1).sum())
print('Decision: se conservan los outliers para no perder muestras de la clase minoritaria.')

### Estandarización z-score

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

stats_post = X_scaled_df.describe().T[['mean', 'std', 'min', 'max']]
stats_post.columns = ['Media', 'Std', 'Min', 'Max']
print('=== Verificacion post-estandarizacion ===')
print(stats_post.round(4).to_string())
print(f'\nMedia global : {X_scaled_df.values.mean():.6f}  (esperado ~0)')
print(f'Std global   : {X_scaled_df.values.std():.6f}   (esperado ~1)')

### Comparación antes vs. después de estandarizar

In [ ]:
sample_feats = feature_cols[:6]

fig, axes = plt.subplots(2, 6, figsize=(22, 7))

for i, feat in enumerate(sample_feats):
    sns.histplot(X[feat], ax=axes[0, i], kde=True, color='#3498db', bins=30, alpha=0.7)
    axes[0, i].set_title(feat[:18], fontsize=9)
    axes[0, i].set_xlabel('')
    if i == 0:
        axes[0, i].set_ylabel('Antes (escala original)', fontsize=9)

    sns.histplot(X_scaled_df[feat], ax=axes[1, i], kde=True, color='#e67e22', bins=30, alpha=0.7)
    axes[1, i].set_xlabel('')
    if i == 0:
        axes[1, i].set_ylabel('Despues (z-score)', fontsize=9)

plt.suptitle('Distribucion de features: antes vs. despues de estandarizacion',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Impacto en distancias euclidianas

In [ ]:
from sklearn.metrics import pairwise_distances

rng = np.random.default_rng(42)
idx = rng.choice(len(X), size=200, replace=False)

dist_before = pairwise_distances(X.values[idx], metric='euclidean').flatten()
dist_after  = pairwise_distances(X_scaled[idx],  metric='euclidean').flatten()
dist_before = dist_before[dist_before > 0]
dist_after  = dist_after[dist_after  > 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(dist_before, bins=60, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].set_title('Distancias euclidianas — sin estandarizar')
axes[0].set_xlabel('Distancia')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(dist_after, bins=60, color='#e67e22', alpha=0.8, edgecolor='white')
axes[1].set_title('Distancias euclidianas — estandarizadas')
axes[1].set_xlabel('Distancia')

plt.suptitle('Efecto de la estandarizacion sobre las distancias (n=200 muestras)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Sin estandarizar — media: {dist_before.mean():.2f}, rango: [{dist_before.min():.2f}, {dist_before.max():.2f}]')
print(f'Estandarizadas   — media: {dist_after.mean():.2f},  rango: [{dist_after.min():.2f}, {dist_after.max():.2f}]')

### Construcción del conjunto semi-supervisado

Label Propagation y Label Spreading requieren que la mayoría de muestras tengan `y = -1` (no etiquetadas). Se simula un escenario donde solo el **20% de las muestras tienen etiqueta conocida**, con estratificación para mantener el balance de clases.

In [ ]:
LABELED_RATIO = 0.20
RANDOM_STATE  = 42

X_lab, X_unlab, y_lab, y_unlab = train_test_split(
    X_scaled, y.values,
    test_size=1 - LABELED_RATIO,
    stratify=y.values,
    random_state=RANDOM_STATE
)

y_semi = np.concatenate([y_lab, np.full(len(y_unlab), -1)])
X_semi = np.vstack([X_lab, X_unlab])

print('=== Configuracion semi-supervisada ===')
print(f'Total muestras       : {len(y_semi)}')
print(f'Etiquetadas  (20%)   : {(y_semi != -1).sum()} '
      f'[{(y_semi[y_semi != -1] == 0).sum()} bad, '
      f'{(y_semi[y_semi != -1] == 1).sum()} good]')
print(f'No etiquetadas (80%) : {(y_semi == -1).sum()}')

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
X_semi_pca = pca2.fit_transform(X_semi)

labeled_mask   = y_semi != -1
unlabeled_mask = y_semi == -1

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(X_semi_pca[unlabeled_mask, 0], X_semi_pca[unlabeled_mask, 1],
           c='lightgray', alpha=0.4, s=20, label='No etiquetado', zorder=1)

colors_lab = np.where(y_semi[labeled_mask] == 0, '#e74c3c', '#2ecc71')
ax.scatter(X_semi_pca[labeled_mask, 0], X_semi_pca[labeled_mask, 1],
           c=colors_lab, s=55, edgecolors='black', linewidths=0.5, zorder=2)

legend_elements = [
    Patch(facecolor='lightgray', label=f'No etiquetado ({unlabeled_mask.sum()})'),
    Patch(facecolor='#e74c3c',   label=f'Bad etiquetado ({(y_semi[labeled_mask]==0).sum()})'),
    Patch(facecolor='#2ecc71',   label=f'Good etiquetado ({(y_semi[labeled_mask]==1).sum()})')
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Conjunto semi-supervisado — 20% etiquetado (PCA)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Resumen final del dataset preprocesado

In [ ]:
print('=' * 52)
print('  RESUMEN — Dataset preprocesado (Fase 2)')
print('=' * 52)
print(f'  Dataset            : Red Wine Quality (UCI)')
print(f'  Muestras totales   : {X_semi.shape[0]}')
print(f'  Features           : {X_semi.shape[1]} (todas numericas)')
print(f'  Clases             : 2  (0=bad <7, 1=good >=7)')
print(f'  Valores faltantes  : 0')
print(f'  Encoding           : No requerido')
print(f'  Outliers           : conservados (clase minoritaria)')
print(f'  Escalado           : StandardScaler (z-score)')
print(f'  Etiquetadas  (20%) : {(y_semi != -1).sum()}')
print(f'  No etiquetadas(80%): {(y_semi == -1).sum()}')
print('=' * 52)
print('  Listo para Label Propagation / Label Spreading OK')
print('=' * 52)